# Sentiment Analysis - LSTM (Long Short Term Memory)

### Import Package

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

### Load Dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [3]:
data = pd.read_csv("C:/Users/acer/IMDB Dataset.csv")
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
data.shape

(50000, 2)

In [5]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [6]:
data["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

### One Hot Encoding

In [7]:
data.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace=True)
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [8]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [9]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [10]:
train_data.shape

(40000, 2)

In [11]:
test_data.shape

(10000, 2)

In [12]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data["review"])

x_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [13]:
x_train

array([[1935,    1, 1200, ...,  205,  351, 3856],
       [   3, 1651,  595, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [14]:
x_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [  12,  162,   59, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]], dtype=int32)

In [15]:
y_train = train_data['sentiment']
y_test = test_data['sentiment']

In [16]:
y_train

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64

In [17]:
y_test

33553    1
9427     1
199      0
12447    1
39489    0
        ..
28567    0
25079    1
18707    1
15200    0
5857     1
Name: sentiment, Length: 10000, dtype: int64

### Model Building

In [18]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(units=128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [20]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [22]:
model.fit(x_train, y_train, batch_size=64, epochs=5, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 262s 515ms/step - accuracy: 0.7857 - loss: 0.4570 - val_accuracy: 0.8600 - val_loss: 0.3372
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 239s 478ms/step - accuracy: 0.8513 - loss: 0.3538 - val_accuracy: 0.8462 - val_loss: 0.3636
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 255s 509ms/step - accuracy: 0.8530 - loss: 0.3474 - val_accuracy: 0.8360 - val_loss: 0.3826
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 241s 482ms/step - accuracy: 0.8836 - loss: 0.2886 - val_accuracy: 0.8709 - val_loss: 0.3182
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 240s 479ms/step - accuracy: 0.8990 - loss: 0.2539 - val_accuracy: 0.8610 - val_loss: 0.3401


In [25]:
loss, accuracy = model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 34ms/step - accuracy: 0.8740 - loss: 0.3165


In [26]:
print(loss)

0.3165127635002136


In [27]:
print(accuracy)

0.8740000128746033


### Building Predictive System

In [28]:
def predictive_system(review):
    sequences = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequences, maxlen=200)
    prediction = model.predict(padded_sequence)
    sentiment  = "Positive" if prediction[0][0] >= 0.5 else "Negative"
    return sentiment

In [29]:
predictive_system("This movie is great!")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 719ms/step


'Positive'

In [30]:
predictive_system("UIB is a great campus and I love it")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step


'Positive'

In [31]:
# Save Model
model.save("tugas.h5")

import joblib
joblib.dump(tokenizer, 'tokenizer.pkl') # dump tokenizer

['tokenizer.pkl']